# 02 — Exploratory Data Analysis

Reproduces proposal figures and adds academic-period and correlation views that inform feature choices.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if (ROOT / "rec_center_utils.py").exists():
    pass
elif (ROOT / "rec_center" / "rec_center_utils.py").exists():
    ROOT = ROOT / "rec_center"
elif (ROOT.parent / "rec_center_utils.py").exists():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("Could not locate rec_center_utils.py")
sys.path.insert(0, str(ROOT))

from rec_center_utils import (
    TRAIN_END,
    VAL_END,
    clean_data,
    data_path,
    figures_path,
    load_clean_data,
    load_raw_data,
    save_clean_data,
)

sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

df = load_clean_data()
df.shape



## Target distribution


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["average_utilization"], bins=50, kde=True, ax=ax)
ax.set_title("Distribution of Average Utilization")
ax.set_xlabel("Average Utilization")
ax.set_ylabel("Count")
fig.savefig(figures_path("average_utilization_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()



## Utilization by hour, weekday, and location


In [ ]:

from rec_center_utils import CORRELATION_LABELS, WEEKDAY_ORDER, weekday_name_series

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
hourly = df.groupby("hour")["average_utilization"].mean().reset_index()
sns.barplot(data=hourly, x="hour", y="average_utilization", ax=axes[0], color="#4C72B0")
axes[0].set_title("Average Utilization by Hour of Day")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Average Utilization")

weekday = (
    df.assign(weekday_name=weekday_name_series(df["timestamp"]))
    .groupby("weekday_name", observed=True)["average_utilization"]
    .mean()
    .reindex(WEEKDAY_ORDER)
    .reset_index()
)
sns.barplot(data=weekday, x="weekday_name", y="average_utilization", ax=axes[1], color="#55A868")
axes[1].set_title("Average Utilization by Day of Week")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Average Utilization")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

loc = df.groupby("location")["average_utilization"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=loc, y="location", x="average_utilization", ax=axes[2], color="#C44E52")
axes[2].set_title("Average Utilization by Location")
axes[2].set_xlabel("Average Utilization")
axes[2].set_ylabel("Location")
fig.tight_layout()
for name, ax in zip(["utilization_by_hour.png", "utilization_by_weekday.png", "utilization_by_location.png"], axes):
    ax.figure.savefig(figures_path(name), dpi=150, bbox_inches="tight")
plt.show()



## Monthly trend and weekday-hour heatmap


In [ ]:

monthly = df.groupby(df["timestamp"].dt.to_period("M"))["average_utilization"].mean()
fig, ax = plt.subplots(figsize=(9, 4))
monthly.plot(ax=ax)
ax.set_title("Monthly Average Utilization Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Average Utilization")
fig.savefig(figures_path("monthly_utilization_trend.png"), dpi=150, bbox_inches="tight")
plt.show()

df_heat = df.assign(weekday_name=weekday_name_series(df["timestamp"]))
pivot = df_heat.pivot_table(
    values="average_utilization",
    index="weekday_name",
    columns="hour",
    aggfunc="mean",
).reindex(WEEKDAY_ORDER)
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax)
ax.set_title("Average Utilization Heatmap by Weekday and Hour")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Day of Week")
fig.savefig(figures_path("weekday_hour_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()



## Additional insights for modeling


In [ ]:

period = pd.Series("In Quarter", index=df.index)
period[df["is_summer"] == 1] = "Summer"
period[df["is_winter_break"] == 1] = "Winter Break"
period_df = df.assign(academic_period=period)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=period_df.groupby("academic_period")["average_utilization"].mean().reset_index(),
    x="academic_period",
    y="average_utilization",
    order=["In Quarter", "Summer", "Winter Break"],
    ax=ax,
    palette="Set2",
)
ax.set_title("Utilization by Academic Period")
ax.set_xlabel("Academic Period")
ax.set_ylabel("Average Utilization")
fig.savefig(figures_path("utilization_by_academic_period.png"), dpi=150, bbox_inches="tight")
plt.show()

corr_cols = ["hour", "day_of_week", "month", "is_weekend", "is_summer", "is_finals_week", "average_utilization"]
corr = df[corr_cols].corr().rename(index=CORRELATION_LABELS, columns=CORRELATION_LABELS)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Feature Correlation Heatmap")
fig.savefig(figures_path("feature_correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()



### EDA takeaways

- Peak crowding occurs on weekday afternoons (especially 4–6 PM).
- Track and 2nd Floor areas run hotter than 1st Floor and Lower Exercise Room.
- Summer and winter break periods are materially quieter.
- Nonlinear time interactions justify tree/boosting models over a simple linear baseline.
